In [ ]:
from math import log

In [ ]:
log(1.5)

0.4054651081081644

In [ ]:
from math import sqrt

In [ ]:
sqrt(1+(1.4054)**2)

1.7248620698479054

In [ ]:
1.4054/1.7248

0.8148191094619666

In [ ]:
1/1.7248

0.5797773654916512

In [ ]:
sqrt((0.8148)**2+(0.5797)**2)

0.9999755647014581

In [ ]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

In [ ]:
doc_raw = ["movies is good","movie is bad"]
y = ["positive","negative"]

In [ ]:
nlp = spacy.load("en_core_web_sm")
def preprocess(text):
  doc = nlp(text)
  return [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop]

In [ ]:
docs_tokens = [preprocess(t) for t in doc_raw]
docs_tokens

[['movie', 'good'], ['movie', 'bad']]

In [ ]:
docs_clean = [' '.join(toks) for toks in docs_tokens]
docs_clean

['movie good', 'movie bad']

In [ ]:
vocab = ['bad','good','movie']

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(vocab)

In [ ]:
print("features",vectorizer.get_feature_names_out())
print("idf",vectorizer.idf_)

features ['bad' 'good' 'movie']
idf [1.69314718 1.69314718 1.69314718]


In [ ]:
X = X.toarray()

In [ ]:
X

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
data = [
  {'message':"movie is good","sentiment":"positive"},
  {'message':"movie is bad","sentiment":"negative"}

]

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df

,message,sentiment
0,movie is good,positive
1,movie is bad,negative


In [ ]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [ ]:
df

,message,sentiment
0,movie is good,1
1,movie is bad,0


In [ ]:
def preprocess(text):
  doc = nlp(text.lower())
  tokens = [token.lemma_ for token in doc if not token.is_punct and not token.is_stop]
  return ' '.join(tokens)

In [ ]:
df['tokens'] = df['message'].apply(preprocess)
df

,message,sentiment,tokens
0,movie is good,1,movie good
1,movie is bad,0,movie bad


In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['tokens'])
print(vectorizer.get_feature_names_out())
print(X.toarray())

['bad' 'good' 'movie']
[[0.         0.81480247 0.57973867]
 [0.81480247 0.         0.57973867]]


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lg = LogisticRegression()
lg.fit(X,y)

LogisticRegression()

In [ ]:
test_message = 'movie is not good'
test_message = preprocess(test_message)
test_message

'movie good'

In [ ]:
test_data = vectorizer.transform([test_message])
test_data

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 3)>

In [ ]:
lg.predict(test_data)

array(['positive'], dtype='<U8')

In [ ]:
#import all the libraries
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/spam.csv", encoding='latin-1')
df.head()

,type,text,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df = df[['text','type']]

In [ ]:
df.head()

,text,type
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [ ]:
df.dtypes

,0
text,object
type,object


In [ ]:
df.isna().sum()

,0
text,0
type,0


In [ ]:
countvec = CountVectorizer(stop_words='english',strip_accents='unicode',max_features=1000)

In [ ]:
convert = countvec.fit_transform(df.text)

In [ ]:
X_train = convert.toarray()
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
'''
X = convert.toarray()
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
'''

In [ ]:
y_train = df.type.values
y_train

array(['ham', 'ham', 'spam', ..., 'ham', 'ham', 'ham'], dtype=object)

In [ ]:
lg = LogisticRegression()

In [ ]:
lg.fit(X_train,y_train)

LogisticRegression()

In [ ]:
lg.score(X_train,y_train)

0.9896133871898442

In [ ]:
test_message = ["Customer service annoncement. You have a New Years delivery waiting for you. Please call 07046744435 now to arrange delivery"]

In [ ]:
test_convert = countvec.transform(test_message)

In [ ]:
lg.predict(test_convert)

array(['spam'], dtype=object)

In [ ]:
msg = ["Sorry, I'll call later in meeting."]

In [ ]:
lg.predict(countvec.transform(msg))

array(['ham'], dtype=object)